# KNNMultiRoundRouter - Training

This notebook demonstrates how to train the **KNNMultiRoundRouter**.

## Overview

KNNMultiRoundRouter extends the KNNRouter with a multi-round pipeline:
1. **Decompose**: Break down complex queries into sub-queries
2. **Route**: Use KNN to route each sub-query to the best model
3. **Execute**: Call APIs to get responses from routed models
4. **Aggregate**: Combine sub-query responses into final answer

**Key Features**:
- KNN-based routing (same as single-round KNNRouter)
- Multi-round decomposition and aggregation
- Supports both local LLM (vLLM) and API-based decomposition
- Configurable K value and distance metrics

## 1. Environment Setup

In [1]:
# For Google Colab: Clone repository and install dependencies
import os
# Install required packages (for Colab)
!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter
!pip install -e .
!pip install  pyyaml scikit-learn transformers torch


Cloning into 'LLMRouter'...
remote: Enumerating objects: 6017, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 6017 (delta 98), reused 87 (delta 81), pack-reused 5845 (from 2)
Receiving objects: 100% (6017/6017), 89.41 MiB | 50.95 MiB/s, done.
Resolving deltas: 100% (2946/2946), done.
Updating files: 100% (288/288), done.
/home/zhongjie/LLMRouter
Obtaining file:///home/zhongjie/LLMRouter
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llmrouter-lib (pyproject.toml) ... done
  Created wheel for llmrouter-lib: filename=llmrouter_lib-0.2.0-0.editable-py3-none-any.whl size=15709 sha256=08cf66436fc43f6442e1ad3868c6d8b0ef18961954fa2f170263e563fab8d7ca
  Stored in directory: /tmp/pip-ephem-wheel-cache-03ewvlf_/wheels/82/4a/fd/59c4aec93c356c380d

In [2]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [3]:
from llmrouter.models.knnmultiroundrouter import KNNMultiRoundRouter, KNNMultiRoundRouterTrainer
from llmrouter.utils import setup_environment

setup_environment()
print("Environment setup complete!")

Environment setup complete!


## 2. Configuration

KNNMultiRoundRouter uses the following configuration parameters:

### KNN Parameters

| Parameter | Description | Default |
|-----------|-------------|--------|
| `n_neighbors` | Number of neighbors (K value) | 5 |
| `weights` | Weight function: "uniform" or "distance" | "uniform" |
| `algorithm` | KNN algorithm: "auto", "ball_tree", "kd_tree", "brute" | "auto" |
| `metric` | Distance metric | "minkowski" |
| `p` | Power for Minkowski (1=Manhattan, 2=Euclidean) | 2 |

### Multi-Round Parameters

| Parameter | Description | Default |
|-----------|-------------|--------|
| `base_model` | LLM for decomposition/aggregation | "Qwen/Qwen2.5-3B-Instruct" |
| `use_local_llm` | Use vLLM for local inference | false |
| `api_endpoint` | API endpoint for execution | - |

In [4]:
import yaml

CONFIG_PATH = "configs/model_config_train/knnmultiroundrouter.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print("Current Configuration:")
print("=" * 50)
print(yaml.dump(config, default_flow_style=False))

Current Configuration:
api_endpoint: https://integrate.api.nvidia.com/v1
api_key: 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
base_model: qwen/qwen2.5-7b-instruct
data_path:
  llm_data: data/example_data/llm_candidates/default_llm.json
  llm_embedding_data: data/example_data/llm_candidates/default_llm_embeddings.json
  query_data_test: data/example_data/query_data/default_query_test.jsonl
  query_data_train: data/example_data/query_data/default_query_train.jsonl
  query_embedding_data: data/example_data/routing_data/query_embeddings_longformer.pt
  routing_data_test: data/example_data/routing_data/default_routing_test_data.jsonl
  routing_data_train: data/example_data/routing_data/default_routing_train_data.jsonl
hparam:
  algorithm: auto
  leaf_size: 30
  metric: minkowski
  n_jobs: -1
  n_neighbors: 5
  p: 2
  weights: uniform
metric:
  weights:
    cost: 0
    llm_judge: 0
    performance: 1
model_path:
  ini_model_path: ''
  save_model_path: saved_models/

## 3. Initialize Router

In [5]:
router = KNNMultiRoundRouter(yaml_path=CONFIG_PATH)

print("Router initialized successfully!")
print(f"Number of training samples: {len(router.routing_data_train)}")
print(f"Number of LLM candidates: {len(router.llm_data)}")
print(f"K value (n_neighbors): {config['hparam']['n_neighbors']}")

✅ MetaRouter initialized successfully (YAML + data loaded).
Router initialized successfully!
Number of training samples: 50544
Number of LLM candidates: 7
K value (n_neighbors): 5


## 4. Training

Training fits the KNN model on query embeddings and their best LLM labels.

In [6]:
import torch

device = (
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

trainer = KNNMultiRoundRouterTrainer(router=router, device='device')

print("Trainer initialized!")
print(f"Using device: {device}")

[KNNMultiRoundRouterTrainer] Initialized with router.
Trainer initialized!
Using device: cuda


In [7]:
print("Starting training...")
print("=" * 50)

trainer.train()

print("=" * 50)
print("Training completed!")

Starting training...
Training KNN model on 5608 examples...
KNN model training completed!
Saving trained model to: /home/zhongjie/LLMRouter/llmrouter/saved_models/knnmultiroundrouter/knnmultiroundrouter.pkl
Successfully saved pickle model: /home/zhongjie/LLMRouter/llmrouter/saved_models/knnmultiroundrouter/knnmultiroundrouter.pkl
Model saved successfully!
Training completed!


## 5. Hyperparameter Tuning

In [8]:
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

# Prepare data for hyperparameter tuning
X_train = router.query_embedding_train.numpy() if hasattr(router.query_embedding_list, 'numpy') else router.query_embedding_list
y_train = router.model_name_list

# Test different K values
k_values = [1, 3, 5, 7, 9, 11]
print("K Value Comparison (5-fold CV):")
print("=" * 40)

best_k = 5
best_score = 0

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy')
    mean_score = scores.mean()
    print(f"K={k}: Accuracy = {mean_score:.4f} (+/- {scores.std():.4f})")
    
    if mean_score > best_score:
        best_score = mean_score
        best_k = k

print(f"\nBest K: {best_k} with accuracy: {best_score:.4f}")

K Value Comparison (5-fold CV):
K=1: Accuracy = 0.3607 (+/- 0.0708)
K=3: Accuracy = 0.4158 (+/- 0.0700)
K=5: Accuracy = 0.4399 (+/- 0.0738)
K=7: Accuracy = 0.4542 (+/- 0.0724)
K=9: Accuracy = 0.4672 (+/- 0.0681)
K=11: Accuracy = 0.4725 (+/- 0.0701)

Best K: 11 with accuracy: 0.4725


In [ ]:
# Test different distance metrics
metrics = [('euclidean', 2), ('manhattan', 1), ('minkowski', 3)]

print("\nDistance Metric Comparison:")
print("=" * 40)

for metric_name, p in metrics:
    if metric_name == 'minkowski':
        knn = KNeighborsClassifier(n_neighbors=best_k, metric='minkowski', p=p)
        name = f"Minkowski (p={p})"
    else:
        knn = KNeighborsClassifier(n_neighbors=best_k, metric=metric_name)
        name = metric_name.capitalize()
    
    scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy')
    print(f"{name}: Accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})")

## 6. Model Verification

In [9]:
# Test routing on a sample query (without execution)
# Note: Multi-round routers perform decomposition + routing, not just routing

test_queries = [
    {"query": "What is the capital of France?"},
    {"query": "Explain machine learning in simple terms."},
    {"query": "Calculate the integral of x^2."},
]

print("Test Routing (KNN-based):")
print("=" * 60)

for i, query in enumerate(test_queries, 1):
    # Use the underlying KNN router for simple routing test
    result = router._route_sub_query(query['query'])
    print(f"{i}. {query['query'][:50]}...")
    print(f"   Routed to: {result}")

Test Routing (KNN-based):
Successfully loaded pickle model: /home/zhongjie/LLMRouter/llmrouter/saved_models/knnmultiroundrouter/knnmultiroundrouter.pkl
[KNNMultiRoundRouter] Loaded KNN model from /home/zhongjie/LLMRouter/llmrouter/saved_models/knnmultiroundrouter/knnmultiroundrouter.pkl


Input ids are automatically padded to be a multiple of `config.attention_window`: 512


1. What is the capital of France?...
   Routed to: gemma-2-9b-it
2. Explain machine learning in simple terms....
   Routed to: qwen2.5-7b-instruct
3. Calculate the integral of x^2....
   Routed to: qwen2.5-7b-instruct


## 7. Save Model

In [10]:
import pickle

save_path = config['model_path']['save_model_path']
os.makedirs(os.path.dirname(save_path), exist_ok=True)

with open(save_path, 'wb') as f:
    pickle.dump(router.model, f)

print(f"Model saved to: {save_path}")

Model saved to: saved_models/knnmultiroundrouter/knnmultiroundrouter.pkl


## Summary

In this notebook, we:

1. **Loaded Configuration**: Set up KNNMultiRoundRouter with YAML config
2. **Trained Model**: Fitted KNN classifier on query embeddings
3. **Tuned Hyperparameters**: Tested different K values and distance metrics
4. **Verified Model**: Tested routing on sample queries
5. **Saved Model**: Persisted trained model for inference

**Key Differences from Single-Round KNNRouter**:
- Supports query decomposition into sub-queries
- Aggregates responses from multiple models
- Uses LLM for decomposition and aggregation steps

**Next Steps**:
- Use the next part of notebook for full pipeline inference

# KNNMultiRoundRouter - Inference

This notebook demonstrates how to use a trained **KNNMultiRoundRouter** for inference.

## Pipeline Overview

The multi-round routing pipeline consists of:

1. **Decompose**: Break complex queries into simpler sub-queries using LLM
2. **Route**: Use trained KNN to route each sub-query to the best model
3. **Execute**: Call the selected model API to get responses
4. **Aggregate**: Combine all sub-responses into a final answer

## 1. Environment Setup

In [11]:
from llmrouter.models.knnmultiroundrouter import KNNMultiRoundRouter
from llmrouter.utils import setup_environment
import yaml

setup_environment()

## 2. Load Trained Router

In [12]:
CONFIG_PATH = "configs/model_config_train/knnmultiroundrouter.yaml"

router = KNNMultiRoundRouter(yaml_path=CONFIG_PATH)
print("Router loaded!")

# Load configuration
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print(f"Base model for decomposition: {config.get('base_model', 'Qwen/Qwen2.5-3B-Instruct')}")
print(f"Use local LLM: {config.get('use_local_llm', False)}")

✅ MetaRouter initialized successfully (YAML + data loaded).
Router loaded!
Base model for decomposition: qwen/qwen2.5-7b-instruct
Use local LLM: False


## 3. Simple Query Routing (Chat Mode)

For simple string queries, the router returns just the response.

In [13]:
# Simple chat mode - pass string, get string response
simple_query = "What is the capital of France and what is its population?"

print(f"Query: {simple_query}")
print("=" * 60)

try:
    response = router.route_single(simple_query)
    print(f"\nResponse:\n{response}")
except Exception as e:
    print(f"Error: {e}")
    print("Note: Multi-round routing requires API access for execution.")

Query: What is the capital of France and what is its population?
Successfully loaded pickle model: /home/zhongjie/LLMRouter/llmrouter/saved_models/knnmultiroundrouter/knnmultiroundrouter.pkl
[KNNMultiRoundRouter] Loaded KNN model from /home/zhongjie/LLMRouter/llmrouter/saved_models/knnmultiroundrouter/knnmultiroundrouter.pkl

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Response:
To answer the question "What is the capital of France and what is its population?", let's break it down step by step.

1. **Identifying the Capital of France:**
   - The capital of France is Paris. This is a well-known fact and does not require additional information to confirm.

2. **Finding the Population of Paris:**
   - The population of Paris can vary slightly depending on the source and the time of the latest census. However, as of the most recent data available, Paris has a population of approxima

## 4. Evaluation Mode

For evaluation with metrics, pass a dict with query, task_name, and ground_truth.

In [14]:
# Evaluation mode - pass dict, get detailed result with metrics
eval_query = {
    "query": "What is 15 * 23?",
    "task_name": "math",
    "ground_truth": "345"
}

print(f"Query: {eval_query['query']}")
print(f"Task: {eval_query['task_name']}")
print(f"Ground Truth: {eval_query['ground_truth']}")
print("=" * 60)

try:
    result = router.route_single(eval_query)
    
    print(f"\nResponse: {result.get('response', 'N/A')}")
    print(f"Success: {result.get('success', False)}")
    print(f"Prompt Tokens: {result.get('prompt_tokens', 0)}")
    print(f"Completion Tokens: {result.get('completion_tokens', 0)}")
    if 'task_performance' in result:
        print(f"Task Performance: {result['task_performance']:.2f}")
except Exception as e:
    print(f"Error: {e}")

Query: What is 15 * 23?
Task: math
Ground Truth: 345

Response: The question is asking for the product of 15 and 23.

To solve this, we can break it down as follows:

- First, we can consider 15 * 20, which is 300.
- Then, we can consider 15 * 3, which is 45.
- Adding these two results together: 300 + 45 = 345.

Therefore, the answer is 345.

<answer>345</answer>
Success: True
Prompt Tokens: 271
Completion Tokens: 66
Task Performance: 0.00


## 5. Batch Processing

In [15]:
# Batch processing with multiple queries
batch_queries = [
    {"query": "Explain photosynthesis."},
    {"query": "What causes earthquakes?"},
    {"query": "How do computers work?"},
]

print(f"Processing {len(batch_queries)} queries...")
print("=" * 60)

try:
    results = router.route_batch(batch_queries)
    
    for i, result in enumerate(results, 1):
        print(f"\n{i}. Query: {result.get('query', 'N/A')[:50]}...")
        print(f"   Success: {result.get('success', False)}")
        print(f"   Response: {result.get('response', 'N/A')[:100]}...")
except Exception as e:
    print(f"Error: {e}")

Processing 3 queries...

1. Query: Explain photosynthesis....
   Success: True
   Response: Photosynthesis is a process used by plants, algae, and some bacteria to convert light energy, usuall...

2. Query: What causes earthquakes?...
   Success: True
   Response: To answer the question "What causes earthquakes?", let's break it down step by step:

1. **Understan...

3. Query: How do computers work?...
   Success: True
   Response: To understand how computers work, let's break it down step by step:

1. **Hardware Components**: Com...


## 6. Understanding the Pipeline

Let's examine the multi-round pipeline steps.

In [15]:
# Demonstrate the pipeline components
print("Multi-Round Pipeline Components:")
print("=" * 60)

print("\n1. DECOMPOSITION")
print("   - Uses LLM to break complex query into sub-queries")
print(f"   - Base model: {config.get('base_model', 'Qwen/Qwen2.5-3B-Instruct')}")

print("\n2. ROUTING (KNN-based)")
print(f"   - K value: {config['hparam']['n_neighbors']}")
print(f"   - Distance metric: {config['hparam']['metric']}")
print(f"   - Weight function: {config['hparam']['weights']}")

print("\n3. EXECUTION")
print("   - Calls routed model API for each sub-query")
print(f"   - API endpoint: {config.get('api_endpoint', 'Not configured')}")

print("\n4. AGGREGATION")
print("   - Combines sub-query responses into final answer")
print("   - Uses LLM for intelligent synthesis")

Multi-Round Pipeline Components:

1. DECOMPOSITION
   - Uses LLM to break complex query into sub-queries
   - Base model: qwen/qwen2.5-7b-instruct

2. ROUTING (KNN-based)
   - K value: 5
   - Distance metric: minkowski
   - Weight function: uniform

3. EXECUTION
   - Calls routed model API for each sub-query
   - API endpoint: https://integrate.api.nvidia.com/v1

4. AGGREGATION
   - Combines sub-query responses into final answer
   - Uses LLM for intelligent synthesis


In [16]:
# Show available LLM candidates for routing
print("\nAvailable LLM Candidates:")
print("=" * 60)

for i, (name, info) in enumerate(router.llm_data.items(), 1):
    size = info.get('size', 'unknown')
    print(f"{i}. {name}: {size}B parameters")


Available LLM Candidates:
1. qwen2.5-7b-instruct: 7BB parameters
2. llama-3.1-8b-instruct: 8BB parameters
3. mistral-7b-instruct-v0.3: 7BB parameters
4. llama-3.3-nemotron-super-49b-v1: 49BB parameters
5. llama3-70b-instruct: 70BB parameters
6. mixtral-8x7b-instruct-v0.1: 45BB parameters
7. mixtral-8x22b-instruct-v0.1: 141BB parameters


## 7. Evaluation

In [17]:
from llmrouter.evaluation import evaluate_batch
from collections import Counter

eval_data = []
for r in results:
    if r.get('ground_truth'):
        eval_data.append({
            'prediction': r.get('response', ''),
            'ground_truth': r.get('ground_truth'),
            'metric': r.get('metric', 'em')
        })

if eval_data:
    eval_results = evaluate_batch(eval_data)
    scores = [r['score'] for r in eval_results]
    avg_score = sum(scores) / len(scores) if scores else 0
else:
    eval_results = []
    avg_score = 0


print("\nEvaluation Metrics:")
print("-" * 50)
print(f"Total queries: {len(results)}")
print(f"Evaluated queries: {len(eval_data)}")
print(f"Average score ({eval_data[0]['metric'] if eval_data else 'N/A'}): {avg_score:.4f}")

if eval_results:
    print("\nDetailed Scores:")
    print("-" * 50)
    for i, r in enumerate(eval_results[:5], 1):
        pred = r.get('prediction', '')[:30]
        gt = r.get('ground_truth', '')[:30]
        print(f"  {i}. Score: {r['score']:.4f} | pred: {pred} | ground truth: {gt}")



Evaluation Metrics:
--------------------------------------------------
Total queries: 3
Evaluated queries: 0
Average score (N/A): 0.0000


## 8. File-Based Inference

Load queries from a file and save results.

In [18]:
import json

# Load queries from a JSONL file
def load_queries_from_file(file_path):
    """Load queries from a JSONL file."""
    queries = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                queries.append(json.loads(line))
    return queries

# Save results to a JSONL file
def save_results_to_file(results, output_path):
    """Save routing results to a JSONL file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    print(f"Results saved to: {output_path}")

# Example: Load from default query file
QUERY_FILE = "data/example_data/query_data/default_query_test.jsonl"
OUTPUT_FILE = "outputs/knnmultiroundrouter_results.jsonl"

if os.path.exists(QUERY_FILE):
    # Load queries
    file_queries = load_queries_from_file(QUERY_FILE)
    print(f"Loaded {len(file_queries)} queries from: {QUERY_FILE}")
    
    # Route queries (limit to 5 for demo due to API costs)
    try:
        file_results = router.route_batch(file_queries[:5])
        print(f"Routed {len(file_results)} queries")
        
        # Save results
        save_results_to_file(file_results, OUTPUT_FILE)
        
        # Show sample results
        print(f"\nSample results:")
        for i, result in enumerate(file_results[:3], 1):
            print(f"  {i}. {result.get('query', '')[:40]}...")
            print(f"     Success: {result.get('success', False)}")
    except Exception as e:
        print(f"Error during batch routing: {e}")
else:
    print(f"Query file not found: {QUERY_FILE}")
    print("Create a JSONL file with format: {\"query\": \"Your question\"}")

Loaded 706 queries from: data/example_data/query_data/default_query_test.jsonl

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Routed 5 queries
Results saved to: outputs/knnmultiroundrouter_results.jsonl

Sample results:
  1. Q: There are 4 houses in a row, numbered...
     Success: True
  2. Q: There are 3 houses in a row, numbered...
     Success: False
  3. Q: There are 3 houses in a row, numbered...
     Success: False


## Summary

**KNNMultiRoundRouter** provides:
- Query decomposition for complex questions
- KNN-based routing for each sub-query
- Parallel execution across multiple models
- Intelligent response aggregation

**Use Cases**:
- Complex questions requiring multiple expertise areas
- Multi-step reasoning tasks
- Questions that benefit from specialized models

**Requirements**:
- Trained KNN model (from training notebook)
- API access for LLM execution
- Optional: vLLM for local decomposition/aggregation